In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Carga de datos

Fecha de los documentos asignados: 20250714

In [19]:
volumen = pd.read_csv("https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/84ac5875-4728-4df8-aeca-e4f78e5695b9", delimiter=";")
velocidad = pd.read_csv("https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/0fa5a8b4-821d-443d-8d76-2db8696dae00", delimiter=";", encoding="latin1").iloc[:, :-1]
estaciones = pd.read_csv("https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/d94b5a6f-b707-4dd4-bf08-6fe708afc250", delimiter=";", encoding="latin1", usecols=range(15))

In [20]:
volumen

,Estacion,Fecha,Hora,Carril 1 ligeros,Carril 1 pesados,Carril 2 ligeros,Carril 2 pesados,Carril 3 ligeros,Carril 3 pesados,Carril 4 ligeros,Carril 4 pesados,Carril 5 ligeros,Carril 5 pesados,Carril 6 ligeros,Carril 6 pesados
0,1,14/07/2025,01:00,43,8,0,0,0,0,36,4,16,0,0,0
1,1,14/07/2025,02:00,18,0,0,0,0,0,8,1,6,1,0,0
2,1,14/07/2025,03:00,17,0,0,0,0,0,7,0,2,0,0,0
3,1,14/07/2025,04:00,15,4,0,0,0,0,4,1,4,0,0,0
4,1,14/07/2025,05:00,39,3,0,0,0,0,14,3,9,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12931,9259,20/07/2025,20:00,152,3,0,0,0,0,215,6,0,0,0,0
12932,9259,20/07/2025,21:00,141,7,0,0,0,0,208,10,0,0,0,0
12933,9259,20/07/2025,22:00,104,3,0,0,0,0,196,4,0,0,0,0
12934,9259,20/07/2025,23:00,60,4,0,0,0,0,116,4,0,0,0,0


In [21]:
velocidad

,Fecha,Hora,Sistema,ETD,Detector,0-50 (km/h),50-80 (km/h),80-120 (km/h),120-255 (km/h),Velocidad media (km/h),0-6 (m),6-999 (m),Vehículos totales
0,2025-07-14,01:00:00 - 01:30:00,N-634,"[N-634] 11-ETD SAN SEBASTIÁN-SANTANDER, N-634 ...",Lento SANTANDER,3,15,1,0,"60,53",19,0,19
1,2025-07-14,01:00:00 - 01:30:00,N-634,"[N-634] 11-ETD SAN SEBASTIÁN-SANTANDER, N-634 ...",Lento SAN SEBASTIÁN,3,11,0,0,"56,43",13,1,14
2,2025-07-14,01:00:00 - 01:30:00,N-I,"[N-I] 100-ETD VITORIA-GASTEIZ-LASARTE-ORIA, N-...",Central VITORIA-GASTEIZ,0,0,0,0,"0,0",0,0,0
3,2025-07-14,01:00:00 - 01:30:00,N-I,"[N-I] 100-ETD VITORIA-GASTEIZ-LASARTE-ORIA, N-...",Lento VITORIA-GASTEIZ,0,12,37,2,"93,33",35,16,51
4,2025-07-14,01:00:00 - 01:30:00,N-I,"[N-I] 100-ETD VITORIA-GASTEIZ-LASARTE-ORIA, N-...",Central LASARTE-ORIA,0,0,4,3,"117,14",7,0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
83069,2025-07-20,24:30:00 - 00:00:00,N-636,"[N-636] 133-ETD BIZKAIA-BERGARA, N-636 pk 29,7...",Lento BIZKAIA,0,9,0,0,65,7,2,9
83070,2025-07-20,24:30:00 - 00:00:00,GI-2132,"[GI-2132] 93-ETD ASTIGARRAGA-ERRENTERIA, GI-21...",Lento ASTIGARRAGA,0,10,4,0,75,11,3,14
83071,2025-07-20,24:30:00 - 00:00:00,GI-2132,"[GI-2132] 93-ETD ASTIGARRAGA-ERRENTERIA, GI-21...",Lento ERRENTERIA,0,9,3,0,"73,75",10,2,12
83072,2025-07-20,24:30:00 - 00:00:00,GI-2632,"[GI-2632] 87-ETD BIZKAIA-BEASAIN, GI-2632 pk 5...",Lento BIZKAIA,1,4,1,0,"64,17",6,0,6


In [ ]:
velocidad["Hora"] = velocidad["Hora"].str.split('-').str[0].str[:3] +"00"
velocidad["Hora"] = velocidad["Hora"].astype(str).str.strip()

velocidad["Velocidad media (km/h)"] = velocidad["Velocidad media (km/h)"].str.replace(',', '.')
velocidad["Velocidad media (km/h)"] = pd.to_numeric(velocidad["Velocidad media (km/h)"])



In [41]:
df_vel_agrupado = velocidad.groupby(["Fecha", "Hora", "Detector", "ETD"]).agg({
    "0-50 (km/h)": "sum",
    "50-80 (km/h)": "sum",
    "80-120 (km/h)": "sum",
    "120-255 (km/h)": "sum",
    "0-6 (m)": "sum",
    "6-999 (m)": "sum",
    "Vehículos totales": "sum",
    "Velocidad media (km/h)": "mean"
}).reset_index()

In [42]:
df_vel_agrupado["Estacion"] = df_vel_agrupado["ETD"].str.split(" ").str[1].str.split("-").str[0]
df_vel_agrupado["Fecha"] = pd.to_datetime(df_vel_agrupado["Fecha"])

In [45]:
df_vel_agrupado

,Fecha,Hora,Detector,ETD,0-50 (km/h),50-80 (km/h),80-120 (km/h),120-255 (km/h),0-6 (m),6-999 (m),Vehículos totales,Velocidad media (km/h),Estacion
0,2025-07-14,01:00,Central A GI-20 AÑORGA,[GI-20-10] 34-ETD SAN SEBASTIÁN-A GI-20 AÑORGA...,0,6,5,0,11,0,11,80.750,34
1,2025-07-14,01:00,Central A LA GI-20,"[GI-11] 247-ETD N-I-A LA GI-20, GI-11 pk 0,000...",3,51,40,0,82,12,94,78.630,247
2,2025-07-14,01:00,Central A LA GI-20,"[GI-40] 316-ETD ENLACE HOSPITAL-A LA GI-20, GI...",0,0,5,0,0,5,5,100.000,316
3,2025-07-14,01:00,Central A LA GI-41,"[GI-40] 287-ETD ROTONDA GARBERA-A LA GI-41, GI...",0,2,10,0,12,0,12,94.170,287
4,2025-07-14,01:00,Central A-15,"[GI-41] 302-ETD SAN SEBASTIÁN-A-15, GI-41 pk 0...",0,0,0,5,4,1,5,140.000,302
...,...,...,...,...,...,...,...,...,...,...,...,...,...
41532,2025-07-20,24:00,"Rápido USURBIL, AP-8","[GI-20] 116-ETD ERRENTERIA AP-8-USURBIL, AP-8,...",0,9,43,3,51,4,55,96.435,116
41533,2025-07-20,24:00,"Rápido USURBIL, AP-8","[GI-20] 117-ETD ERRENTERIA AP-8-USURBIL, AP-8,...",0,43,63,0,106,0,106,85.805,117
41534,2025-07-20,24:00,"Rápido USURBIL, AP-8","[GI-20] 249-ETD USURBIL, AP-8-ERRENTERIA AP-8,...",0,0,29,4,32,1,33,104.855,249
41535,2025-07-20,24:00,"Rápido USURBIL, AP-8","[GI-20] 250-ETD USURBIL, AP-8-ERRENTERIA AP-8,...",0,0,29,4,32,1,33,104.855,250


In [25]:
volumen["Fecha"] = volumen["Fecha"].str.strip()
volumen["Fecha"] = pd.to_datetime(volumen["Fecha"], format="%d/%m/%Y")

In [26]:
volumen["Estacion"] = volumen["Estacion"].astype(str).str.strip()
volumen["Hora"] = volumen["Hora"].astype(str).str.strip()

In [47]:
df_vel_vol = pd.merge(df_vel_agrupado, volumen, on=["Fecha", "Estacion", "Hora"])
df_vel_vol.columns

Index(['Fecha', 'Hora', 'Detector', 'ETD', '0-50 (km/h)', '50-80 (km/h)',
       '80-120 (km/h)', '120-255 (km/h)', '0-6 (m)', '6-999 (m)',
       'Vehículos totales', 'Velocidad media (km/h)', 'Estacion',
       'Carril 1 ligeros ', 'Carril 1 pesados', 'Carril 2 ligeros ',
       'Carril 2 pesados', 'Carril 3 ligeros ', 'Carril 3 pesados',
       'Carril 4 ligeros ', 'Carril 4 pesados', 'Carril 5 ligeros ',
       'Carril 5 pesados', 'Carril 6 ligeros ', 'Carril 6 pesados'],
      dtype='object')

In [35]:
estaciones

,System,ETD code,System code,Description,Country code,Country,Municipality code,Municipality,Territory code,Territory,Postal code,GPSX,GPSY,X,Y
0,GI-2132,1,1,"[GI-2132] 1-ETD LASARTE-ORIA-ASTIGARRAGA, GI-...",108,España,,,20,Gipuzkoa,,580428.65,4792191.27,-2.008765,43.278246
1,GI-636,3,1,"[GI-636] 3-ETD PASAIA-IRUN, GI-636 pk 7,900, ...",108,España,53,LEZO,20,Gipuzkoa,20100,594584.89,4798000.13,-1.833330,43.328896
2,N-634,11,1,"[N-634] 11-ETD SAN SEBASTIÁN-SANTANDER, N-634...",108,España,30,EIBAR,20,Gipuzkoa,20600,545592.28,4782339.10,-2.438889,43.192449
3,N-I,12,1,"[N-I] 12-ETD VITORIA-GASTEIZ-LASARTE-ORIA, N-...",108,España,71,TOLOSA,20,Gipuzkoa,20400,575944.00,4778800.00,-2.065870,43.158142
4,GI-20,16,1,"[GI-20] 16-ETD USURBIL, AP-8-ERRENTERIA,AP-8,...",108,España,69,DONOSTIA-SAN SEBASTIAN,20,Gipuzkoa,0,583607.00,4795202.00,-1.969142,43.305006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,GI-631,9157,1,"[GI-631] 9157-ETD ZUMARRAGA-ZUMAIA, GI-631 pk...",108,España,27,ZESTOA,20,Gipuzkoa,20740,559996.77,4788856.73,-2.260914,43.250126
81,GI-2632,9159,1,"[GI-2632] 9159-ETD BEASAIN-BIZKAIA, GI-2632 p...",108,España,77,URRETXU,20,Gipuzkoa,20700,553008.18,4770764.28,-2.348733,43.087742
82,GI-627,9211,1,"[GI-627] 9211-ETD ARABA-EIBAR, GI-627 pk 51,4...",108,España,74,BERGARA,20,Gipuzkoa,20570,547942.85,4779948.64,-2.410168,43.170779
83,GI-2630,9259,1,"[GI-2630] 9259-ETD URRETXU-BERGARA, GI-2630 p...",108,España,51,LEGAZPI,20,Gipuzkoa,20230,554510.32,4769098.80,-2.330442,43.072638


In [29]:
estaciones["ETD code"]

0         1
1         3
2        11
3        12
4        16
      ...  
80     9157
81     9159
82     9211
83     9259
84    10169
Name: ETD code, Length: 85, dtype: int64